# P1 · Dữ liệu & điều phối — Bước 1 (+0)

**Input (đọc):** data/raw/*.csv (M5 thô)

**Output (bạn phải tạo, đúng format):** common/sample_ids.csv, common/segment_labels.parquet, data/processed/data_long.parquet

**Code mẫu chạy được:** mở `notebooks/M5_reference.ipynb` phần **§1** — copy các cell sang đây rồi mở rộng.

> Quy tắc vàng: không sửa `common/`; không shuffle; tôn trọng trục thời gian; ML dùng lag ≥ 28. Forecast lưu vào `data/outputs/forecasts/<model>.parquet` với cột: `series_id,date,d_int,quantile_level,value,model`.

In [ ]:
import sys, subprocess, importlib, time, warnings, os, json, math, random
# --- 0.2 Import lõi + dò package nào dùng được ------------------------------
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
np.random.seed(42); random.seed(42)
plt.rcParams["figure.figsize"] = (9, 4); plt.rcParams["axes.grid"] = True

def can_import(mod):
    try:
        importlib.import_module(mod); return True
    except Exception:
        return False

AVAILABLE = {
    "pyarrow":      can_import("pyarrow"),
    "lightgbm":     can_import("lightgbm"),
    "statsforecast":can_import("statsforecast"),
    "chronos":      can_import("chronos"),
    "torch":        can_import("torch"),
    "ttm":          can_import("tsfm_public"),
}
try:
    import torch
    AVAILABLE["cuda"] = torch.cuda.is_available()
except Exception:
    AVAILABLE["cuda"] = False

print("Package/phần cứng dùng được:")
for k, v in AVAILABLE.items():
    print(f"  {'✓' if v else '✗'} {k}: {v}")
PARQUET_ENGINE = "pyarrow" if AVAILABLE["pyarrow"] else None

In [ ]:
# --- Nạp THAM SỐ CỐ ĐỊNH từ common/config.yaml (nguồn chân lý duy nhất) -------
import yaml
from pathlib import Path
_c = yaml.safe_load(open("common/config.yaml", encoding="utf-8"))
CONFIG = dict(
    SEED=_c["seed"], HORIZON=_c["horizon"],
    SAMPLE_PER_SEGMENT=_c.get("sampling", {}).get("cap", 1000),
    TRAIN_END=_c["split"]["train_end"], VAL_END=_c["split"]["val_end"],
    TEST_END=_c["split"]["test_end"],
    QUANTILES=_c["quantiles"], ADI_TH=_c["sbc"]["adi_threshold"],
    CV2_TH=_c["sbc"]["cv2_threshold"],
    CHRONOS_MODEL=_c["foundation"]["chronos_model"],
    CHRONOS_CONTEXT=_c["foundation"]["chronos_context"],
    TTM_MODEL=_c["foundation"]["ttm_model"],
    TTM_CONTEXT=_c["foundation"]["ttm_context"],
)
CONFIG["TEST_START"] = CONFIG["VAL_END"] + 1
Q = CONFIG["QUANTILES"]
RAW = Path("data/raw"); OUT = Path("data/outputs"); FC = OUT / "forecasts"
OUT.mkdir(parents=True, exist_ok=True); FC.mkdir(parents=True, exist_ok=True)
PARQUET_ENGINE = "pyarrow"
def save_parquet(df, path):
    path = Path(path); df.to_parquet(path, index=False); return path
print("CONFIG nạp từ common/config.yaml:", {k: CONFIG[k] for k in ["SEED","HORIZON","TRAIN_END","VAL_END","TEST_END"]})

## Đọc thô → ADI/CV² → nhãn SBC → mẫu → long

In [ ]:
# --- 1.1 Đọc dữ liệu thô -----------------------------------------------------
t0 = time.time()
calendar = pd.read_csv(RAW / "calendar.csv")
prices   = pd.read_csv(RAW / "sell_prices.csv")
sales    = pd.read_csv(RAW / "sales_train_evaluation.csv")   # d_1..d_1941

ID_COLS  = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
D_COLS   = [c for c in sales.columns if c.startswith("d_")]
N_DAYS   = len(D_COLS)
print(f"sales: {sales.shape} | ngày lịch sử: {N_DAYS} (d_1..d_{N_DAYS})")
print(f"calendar: {calendar.shape} | prices: {prices.shape}")
print(f"Số chuỗi (item×store): {sales['id'].nunique():,}")
print(f"Đọc xong trong {time.time()-t0:.1f}s")
sales.head(3)

In [ ]:
# --- 1.2 schema: khai báo cột & ngày split (thay cho src/common/schema.py) ---
schema = dict(
    id_cols   = ID_COLS,
    target    = "sales",
    time_key  = "d",
    date_col  = "date",
    train_days = [f"d_{i}" for i in range(1, CONFIG['TRAIN_END']+1)],
    val_days   = [f"d_{i}" for i in range(CONFIG['TRAIN_END']+1, CONFIG['VAL_END']+1)],
    test_days  = [f"d_{i}" for i in range(CONFIG['VAL_END']+1, CONFIG['TEST_END']+1)],
)
print("train:", schema['train_days'][0], "->", schema['train_days'][-1],
      "| val:", schema['val_days'][0], "->", schema['val_days'][-1],
      "| test:", schema['test_days'][0], "->", schema['test_days'][-1])
assert len(schema['test_days']) == CONFIG['HORIZON']

In [ ]:
# --- 1.3 ADI & CV^2 trên ma trận wide (chỉ dùng train để tránh rò rỉ) --------
train_mat = sales[schema['train_days']].to_numpy(dtype=np.float32)   # (n_series, n_train_days)
T = train_mat.shape[1]

nonzero_mask   = train_mat > 0
n_nonzero      = nonzero_mask.sum(axis=1)                            # số kỳ có nhu cầu
sum_nz         = np.where(nonzero_mask, train_mat, 0).sum(axis=1)
mean_nz        = np.divide(sum_nz, n_nonzero, out=np.zeros_like(sum_nz), where=n_nonzero>0)
# std của các giá trị khác 0
sq = np.where(nonzero_mask, train_mat**2, 0).sum(axis=1)
var_nz = np.divide(sq, n_nonzero, out=np.zeros_like(sq), where=n_nonzero>0) - mean_nz**2
var_nz = np.clip(var_nz, 0, None)
std_nz = np.sqrt(var_nz)

with np.errstate(divide="ignore", invalid="ignore"):
    ADI = np.where(n_nonzero > 0, T / n_nonzero, np.inf)
    CV2 = np.where(mean_nz > 0, (std_nz / mean_nz) ** 2, 0.0)

seg_df = sales[ID_COLS].copy()
seg_df["series_id"] = seg_df["id"]
seg_df["n_nonzero"] = n_nonzero
seg_df["zero_rate"] = 1 - n_nonzero / T
seg_df["ADI"] = ADI
seg_df["CV2"] = CV2
print("Tính ADI/CV^2 xong.")
seg_df[["series_id","zero_rate","ADI","CV2"]].describe()

In [ ]:
# --- 2.1 Áp ngưỡng SBC -> nhãn nhóm -----------------------------------------
adi_th, cv2_th = CONFIG["ADI_TH"], CONFIG["CV2_TH"]
cond = [
    (seg_df.ADI <  adi_th) & (seg_df.CV2 <  cv2_th),   # smooth
    (seg_df.ADI <  adi_th) & (seg_df.CV2 >= cv2_th),   # erratic
    (seg_df.ADI >= adi_th) & (seg_df.CV2 <  cv2_th),   # intermittent
    (seg_df.ADI >= adi_th) & (seg_df.CV2 >= cv2_th),   # lumpy
]
seg_df["segment"] = np.select(cond, ["smooth","erratic","intermittent","lumpy"], default="intermittent")

summary = (seg_df.groupby("segment")
           .agg(n_series=("series_id","size"),
                zero_rate=("zero_rate","mean"),
                ADI=("ADI", lambda s: np.median(s[np.isfinite(s)])),
                CV2=("CV2","median"))
           .reindex(["smooth","erratic","intermittent","lumpy"]))
summary["pct"] = (summary.n_series / summary.n_series.sum() * 100).round(1)
print("Phân bố nhóm demand (toàn bộ 30.490 chuỗi):")
print(summary.round(3))

In [ ]:
# --- Lấy mẫu phân tầng: TOÀN BỘ smooth+erratic, cap intermittent+lumpy -------
rare = {"smooth", "erratic"}
cap = _c["sampling"]["cap"]
picks = []
for seg, g in seg_df.groupby("segment"):
    n = len(g) if seg in rare else min(len(g), cap)
    picks.append(g.sample(n=n, random_state=CONFIG["SEED"]))
sample_ids = sorted(pd.concat(picks)["series_id"].tolist())
seg_labels = seg_df[["series_id","ADI","CV2","zero_rate","segment"]].copy()
Path("common").mkdir(exist_ok=True)
seg_labels.to_parquet("common/segment_labels.parquet", index=False)
pd.DataFrame({"series_id": sample_ids}).to_csv("common/sample_ids.csv", index=False)
print(f"Mẫu: {len(sample_ids)} chuỗi | {seg_df.set_index('series_id').loc[sample_ids,'segment'].value_counts().to_dict()}")

In [ ]:
# --- 2.4 Melt CHỈ các chuỗi mẫu sang long + join calendar/prices ------------
t0 = time.time()
cal_keep = ["d","date","wm_yr_wk","wday","month","year",
            "event_name_1","event_type_1","snap_CA","snap_TX","snap_WI"]
sub = sales[sales["id"].isin(sample_ids)].copy()
long = sub.melt(id_vars=ID_COLS, value_vars=D_COLS, var_name="d", value_name="sales")
long = long.merge(calendar[cal_keep], on="d", how="left")
long = long.merge(prices, on=["store_id","item_id","wm_yr_wk"], how="left")

long["series_id"] = long["id"]
long["date"] = pd.to_datetime(long["date"])
long["d_int"] = long["d"].str.slice(2).astype("int32")
long["is_event"] = long["event_name_1"].notna().astype("int8")
# SNAP theo bang của chuỗi
snap_map = {"CA":"snap_CA","TX":"snap_TX","WI":"snap_WI"}
long["snap"] = 0
for st, col in snap_map.items():
    m = long.state_id == st
    long.loc[m, "snap"] = long.loc[m, col].values
long["snap"] = long["snap"].astype("int8")

# downcast
for c in ["sales"]:
    long[c] = long[c].astype("float32")
long = long.merge(seg_labels[["series_id","segment"]], on="series_id", how="left")
long = long.sort_values(["series_id","d_int"]).reset_index(drop=True)
print(f"data_long: {long.shape}  |  {time.time()-t0:.1f}s")
print(f"Bộ nhớ ~ {long.memory_usage(deep=True).sum()/1e6:.0f} MB")
long.head(3)

In [ ]:
# --- Lưu data_long (mẫu) để P2..P5 dùng chung ------------------------------
Path("data/processed").mkdir(parents=True, exist_ok=True)
long.to_parquet("data/processed/data_long.parquet", index=False)
print("Đã lưu data/processed/data_long.parquet:", long.shape)

In [ ]:
# --- 2.5 Bảng actuals cho test (dùng cho đánh giá) --------------------------
# actual test: các ngày d_TEST_START..d_TEST_END của chuỗi mẫu
test_days = schema["test_days"]
actual_test = (long[long.d.isin(test_days)]
               [["series_id","date","d_int","sales","segment"]]
               .rename(columns={"sales":"actual"})
               .reset_index(drop=True))
# train history (long) để tính scale RMSSE/pinball
train_hist = long[long.d_int <= CONFIG["VAL_END"]][["series_id","d_int","sales"]].copy()
print("actual_test:", actual_test.shape, "| train_hist:", train_hist.shape)

## Bước 4 — Split theo thời gian + CỬA SỔ ROLLING-ORIGIN (≥3)
Trượt điểm gốc lùi `step` ngày, mỗi cửa sổ 1 test 28 ngày, tất cả nằm trong actuals (≤ d_1941).
Lưu `common/rolling_windows.csv` để P3/P4/P5 chạy trên **cùng bộ cửa sổ**.

In [ ]:
# --- Bước 4: sinh cửa sổ rolling-origin + lưu artifact cố định --------------
_sp = _c["split"]; _rl = _c["rolling"]; H = CONFIG["HORIZON"]
val_len = _sp["val_end"] - _sp["train_end"]        # độ dài val giữ cố định
rows = []
for k in range(_rl["n_windows"]):
    test_end   = _sp["test_end"] - k * _rl["step"]
    test_start = test_end - H + 1
    val_end    = test_start - 1
    train_end  = val_end - val_len
    assert train_end >= 1, "Cửa sổ vượt quá lịch sử — giảm n_windows/step trong config"
    assert test_end <= _sp["max_actual_day"], "Cửa sổ vượt actuals có sẵn (>d_1941)"
    assert (test_end - test_start + 1) == H, "Test phải đúng 28 ngày"
    rows.append(dict(window=f"W{k+1}", train_end=train_end, val_end=val_end,
                     test_start=test_start, test_end=test_end))
rolling_windows = pd.DataFrame(rows)
rolling_windows.to_csv("common/rolling_windows.csv", index=False)
print("Cửa sổ rolling-origin (đã lưu common/rolling_windows.csv):")
print(rolling_windows.to_string(index=False))